# 3E Elektro-Optik Kalite Kontrol PoC — EfficientAD-S Anomali Tespiti Eğitimi (Google Colab)

Bu notebook, **Google Colab T4 GPU** üzerinde **Anomalib** kütüphanesini kullanarak **MVTec AD** veri kümesindeki elektro-optik temsilci kategoriler (**`metal_nut`**, **`transistor`**, **`screw`**) için **EfficientAD-S** (Small) anomali tespit modellerini eğitir, test eder, **ONNX** formatına aktarır ve **conformal kalibrasyon** skorlarını (`calib_<kategori>.npy`) üretir.

### İş Hattı Özeti:
1. **GPU & Kurulum:** `nvidia-smi` kontrolü, Anomalib v1.1.1 (OpenVINO'suz hafif kurulum) ve ONNX Runtime kütüphanelerinin yüklenmesi.
2. **Yapılandırma & Veri Kümesi:** MVTec AD kategorilerinin `./datasets/MVTec` dizinine otomatik indirilmesi (`image_size=256`, `train_batch_size=1`).
3. **Model Eğitimi & Değerlendirme:** `EfficientAd(model_size='small')` ve `Engine(max_epochs=70)` ile T4 üzerinde kategori başına ~10-15 dk eğitim. `engine.test` ile görsel AUROC ve F1 skorlarının hesaplanması ve `out/metrics.json` kaydı.
4. **ONNX Dışa Aktarma & Doğrulama:** `engine.export(model, export_type='onnx')` ile dışa aktarma, `onnxruntime` ile girdi/çıktı doğrulaması ve uygulama entegrasyonu için `out/anomaly_<kategori>.json` metaverisinin üretilmesi.
5. **Conformal Kalibrasyon:** Eğitimde kullanılmamış ~20 sağlam görsel üzerinden ONNX çıkarımı yapılarak referans anomali skorlarının `out/calib_<kategori>.npy` (1-D float32) olarak saklanması.
6. **Paketleme & İndirme:** Tüm çıktıların (`out/`) tek bir ZIP dosyası olarak arşivlenip indirilmesi.


## 1. Ortam Kurulumu ve Donanım Kontrolü
Colab çalışma ortamının GPU (T4 veya dengi) ile çalıştığını doğrular ve Anomalib'in kararlı sürümünü (v1.1.1) OpenVINO ekleri olmadan kurar.


In [ ]:
# Check GPU hardware and resources
!nvidia-smi

# Install Anomalib (pinned to stable release 1.1.1 without heavy OpenVINO extras)
# Install onnx and onnxruntime for export verification and calibration inference
!pip install -q "anomalib==1.1.1" onnx onnxruntime pillow numpy

# ==============================================================================
# FALLBACK & API COMPATIBILITY NOTE:
# ------------------------------------------------------------------------------
# Anomalib underwent major architecture updates between v1.x and v2.x:
# 1. Dataset class:
#    - Anomalib 1.x: `from anomalib.data import MVTec`
#    - Anomalib 2.x: `from anomalib.data import MVTecAD`
#    The notebook uses a dynamic try/except import to support both versions seamlessly.
# 2. ExportType enumeration:
#    - Anomalib 1.x: `from anomalib.deploy import ExportType`
#    - Anomalib 2.x: `from anomalib.engine import ExportType` or `from anomalib import ExportType`
#    Handled dynamically with a string fallback ("onnx").
# 3. Model & Engine:
#    - Both 1.x and 2.x support:
#      `from anomalib.models import EfficientAd`
#      `from anomalib.engine import Engine`
# 4. OpenVINO-free installation:
#    By avoiding `anomalib[openvino]`, we prevent Intel OpenVINO binary bloat
#    and dependency collisions on Google Colab T4 and HF Spaces CPU.
# ==============================================================================


## 2. Yapılandırma ve MVTec AD Veri Kümesi Hazırlığı
Bu bölümde çalıştırılacak kategoriler, dosya yolları ve çözünürlük parametreleri belirlenir.
- **`CATEGORIES`**: Varsayılan liste `['metal_nut', 'transistor', 'screw']`. Hızlı deneme yapmak için tek bir kategoriye (örn. `CATEGORIES = ['metal_nut']`) düşürebilirsiniz.
- **`DATASET_ROOT`**: `./datasets/MVTec` (Anomalib otomatik olarak indirip arşivden çıkarır).
- **`IMAGE_SIZE`**: `(256, 256)` (EfficientAD mimarisinin optimize edildiği çözünürlük).
- **`TRAIN_BATCH_SIZE`**: `1` (EfficientAD mimarisi strictly `batch_size=1` gerektirir).


In [ ]:
import os
from pathlib import Path
import shutil
import time
import numpy as np
import torch
from PIL import Image

# -------------------------------------------------------------------------
# Global Parameters & Configuration
# -------------------------------------------------------------------------
# Set categories to train. Users can easily change this to a single item,
# e.g., CATEGORIES = ['metal_nut'] for quick testing.
CATEGORIES = ['metal_nut', 'transistor', 'screw']

# Dataset and Output paths
DATASET_ROOT = Path("./datasets/MVTec")
OUTPUT_DIR = Path("out")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Training resolution and batch parameters
IMAGE_SIZE = (256, 256)       # Standard input resolution for EfficientAD
TRAIN_BATCH_SIZE = 1         # Strict requirement for EfficientAD architecture
EVAL_BATCH_SIZE = 1
NUM_WORKERS = 2              # Optimal for Colab 2-vCPU environment

# Training duration parameters:
# On a Colab T4 GPU:
# - 1 epoch takes ~2-4 seconds (~250-400 images).
# - MAX_EPOCHS = 70 gives ~15k-20k steps, fitting comfortably within 10-15 minutes
#   per category while achieving >98% AUROC.
# - If exact paper reproduction (70,000 steps) is desired, set MAX_STEPS = 70000 (~20 min/category).
MAX_EPOCHS = 70
MAX_STEPS = None             # Set to integer (e.g. 70000) or None to use MAX_EPOCHS

# Device check
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Hardware] PyTorch device: {device}")
if device == "cuda":
    print(f"[Hardware] GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"[Hardware] VRAM Available: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("[Warning] CUDA is not available! Training will run on CPU and be much slower.")

# Dynamic imports across Anomalib 1.x and 2.x
try:
    from anomalib.data import MVTec as MVTecDataModule
    print("[Imports] Loaded MVTec from anomalib.data (Anomalib 1.x API)")
except ImportError:
    try:
        from anomalib.data import MVTecAD as MVTecDataModule
        print("[Imports] Loaded MVTecAD from anomalib.data (Anomalib 2.x API)")
    except ImportError as e:
        raise ImportError(f"Failed to import MVTec / MVTecAD: {e}")

from anomalib.engine import Engine
from anomalib.models import EfficientAd

try:
    from anomalib.deploy import ExportType
except ImportError:
    try:
        from anomalib import ExportType
    except ImportError:
        ExportType = None

# Initialize datamodules and trigger automatic download
datamodules = {}
for cat in CATEGORIES:
    print()
    print(f"--- [DataModule] Initializing MVTec AD for: {cat} ---")
    try:
        dm = MVTecDataModule(
            root=str(DATASET_ROOT),
            category=cat,
            image_size=IMAGE_SIZE,
            train_batch_size=TRAIN_BATCH_SIZE,
            eval_batch_size=EVAL_BATCH_SIZE,
            num_workers=NUM_WORKERS,
        )
        # Trigger download and verification if category files are not yet present
        dm.prepare_data()
        dm.setup()
        datamodules[cat] = dm
        print(f"✓ Category '{cat}' datamodule is ready at: {DATASET_ROOT / cat}")
    except Exception as e:
        print(f"✗ Error initializing datamodule for '{cat}': {e}")


## 3. EfficientAD-S Modeli Eğitimi ve Test Değerlendirmesi
Bu hücrede her bir kategori için:
1. `EfficientAd(model_size='small')` modeli başlatılır.
2. `Engine` oluşturulur (T4 üzerinde kategori başına ~10-15 dk sürecek `MAX_EPOCHS=70`).
3. Model sadece sağlam görüntülerle eğitilir (`engine.fit`).
4. Test kümesi üzerinde değerlendirme yapılarak görsel düzeyinde **AUROC** ve **F1** skorları hesaplanır (`engine.test`).
5. Metrikler ekrana yazdırılır ve `out/metrics.json` dosyasına kaydedilir.
6. Bir kategorideki olası hata diğer kategorileri durdurmasın diye her kategori `try/except` bloğunda çalıştırılır.


In [ ]:
import json
import time

trained_models = {}
trained_engines = {}
all_metrics = {}

for cat in CATEGORIES:
    print()
    print("=" * 70)
    print(f"TRAINING & EVALUATION FOR CATEGORY: {cat}")
    print("=" * 70)
    
    if cat not in datamodules:
        print(f"Skipping category '{cat}' because datamodule failed setup.")
        continue
    
    try:
        start_time = time.time()
        dm = datamodules[cat]
        
        # Instantiate EfficientAd model with 'small' size
        # Try multiple constructor signatures for cross-version compatibility
        try:
            model = EfficientAd(model_size="small")
        except Exception:
            try:
                model = EfficientAd(model_size="s")
            except Exception:
                from anomalib.models.image.efficient_ad.torch_model import EfficientAdModelSize
                model = EfficientAd(model_size=EfficientAdModelSize.S)
        
        print(f"✓ Initialized EfficientAd(model_size='small') for {cat}")
        
        # Configure Anomalib Engine
        engine_kwargs = {
            "default_root_dir": f"./results/{cat}",
            "accelerator": "auto",
            "devices": 1,
        }
        if MAX_STEPS is not None:
            engine_kwargs["max_steps"] = MAX_STEPS
            print(f"Engine configured with max_steps={MAX_STEPS}")
        else:
            engine_kwargs["max_epochs"] = MAX_EPOCHS
            print(f"Engine configured with max_epochs={MAX_EPOCHS}")
        
        engine = Engine(**engine_kwargs)
        
        # Train model
        print(f"Starting training for '{cat}'...")
        engine.fit(model=model, datamodule=dm)
        elapsed_time = time.time() - start_time
        print(f"✓ Training finished for '{cat}' in {elapsed_time:.1f}s ({elapsed_time/60:.2f} min)")
        
        # Evaluate model on test split
        print(f"Evaluating '{cat}' on test dataset...")
        test_results = engine.test(model=model, datamodule=dm)
        print(f"Raw test metrics output: {test_results}")
        
        # Extract metrics
        cat_metrics = {
            "category": cat,
            "train_time_seconds": round(elapsed_time, 2),
            "train_time_minutes": round(elapsed_time / 60, 2),
            "max_epochs": MAX_EPOCHS,
            "max_steps": MAX_STEPS,
        }
        
        if test_results and isinstance(test_results, list) and len(test_results) > 0:
            raw_dict = test_results[0]
            for k, v in raw_dict.items():
                if isinstance(v, (int, float)):
                    cat_metrics[k] = float(v)
                elif hasattr(v, "item"):
                    cat_metrics[k] = float(v.item())
                else:
                    cat_metrics[k] = str(v)
        
        # Identify image AUROC and F1 Score
        img_auroc = (
            cat_metrics.get("image_AUROC") or 
            cat_metrics.get("Image_AUROC") or 
            cat_metrics.get("image_auroc") or 
            cat_metrics.get("test_image_AUROC")
        )
        img_f1 = (
            cat_metrics.get("image_F1Score") or 
            cat_metrics.get("Image_F1Score") or 
            cat_metrics.get("image_f1_score") or 
            cat_metrics.get("test_image_F1Score")
        )
        
        print("-" * 50)
        print(f"EVALUATION SUMMARY [{cat}]:")
        if isinstance(img_auroc, (int, float)):
            print(f"  * Image AUROC : {img_auroc:.4f}")
        else:
            print(f"  * Image AUROC : {img_auroc}")
        if isinstance(img_f1, (int, float)):
            print(f"  * Image F1    : {img_f1:.4f}")
        else:
            print(f"  * Image F1    : {img_f1}")
        print("-" * 50)
        
        all_metrics[cat] = cat_metrics
        trained_models[cat] = model
        trained_engines[cat] = engine
        
    except Exception as e:
        print(f"✗ Training/Evaluation failed for category '{cat}': {e}")
        import traceback
        traceback.print_exc()

# Save metrics to out/metrics.json
metrics_file = OUTPUT_DIR / "metrics.json"
with open(metrics_file, "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, indent=2, ensure_ascii=False)
print()
print(f"✓ Successfully saved all metrics to: {metrics_file}")


## 4. ONNX Formatına Dışa Aktarma, Doğrulama ve Metadata JSON Kaydı
Bu hücrede:
1. Model `engine.export(model, export_type='onnx')` (`ExportType.ONNX`) çağrısı ile dışa aktarılır.
2. Dışa aktarılan ONNX dosyası `out/anomaly_<kategori>.onnx` yoluna kopyalanır.
3. `onnxruntime` ile model yüklenerek girdi ve çıktı tensör adları ile şekilleri doğrulanır.
4. Dummy ve örnek görsel ile çıkarım (inference) testi yapılarak anomali haritası (`anomaly_map`) ve görsel skoru (`image_score`) çıktıları tespit edilir.
5. Uygulamanın (Lane B `engines/anomaly.py`) modeli hatasız çalıştırmasını sağlayan girdi adı, girdi boyutu, önişleme kuralları (resize 256, normalization) ve normalizasyon eşik değerlerini (`image_threshold`, `pixel_threshold`, `min`, `max`) içeren `out/anomaly_<kategori>.json` dosyası oluşturulur.


In [ ]:
import json
from pathlib import Path
import shutil
import numpy as np
import onnxruntime as ort
from PIL import Image

onnx_sessions = {}

for cat in CATEGORIES:
    print()
    print("=" * 70)
    print(f"ONNX EXPORT & VERIFICATION FOR CATEGORY: {cat}")
    print("=" * 70)
    
    if cat not in trained_models:
        print(f"Skipping ONNX export for '{cat}' (model is not available).")
        continue
    
    try:
        model = trained_models[cat]
        engine = trained_engines[cat]
        
        # Determine ExportType
        export_type_arg = ExportType.ONNX if ExportType is not None else "onnx"
        export_temp_dir = Path(f"./results/{cat}/exported")
        export_temp_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"Exporting model to ONNX format for '{cat}'...")
        export_output = engine.export(
            model=model,
            export_type=export_type_arg,
            export_root=export_temp_dir
        )
        print(f"engine.export response: {export_output}")
        
        # Find the generated .onnx file
        exported_onnx_path = None
        if export_output and Path(str(export_output)).is_file():
            exported_onnx_path = Path(str(export_output))
        else:
            candidates = list(export_temp_dir.rglob("*.onnx")) + list(Path(f"./results/{cat}").rglob("*.onnx"))
            if candidates:
                exported_onnx_path = candidates[0]
                
        if exported_onnx_path is None or not exported_onnx_path.exists():
            raise FileNotFoundError(f"Could not locate exported ONNX file for category '{cat}'.")
        
        dst_onnx_path = OUTPUT_DIR / f"anomaly_{cat}.onnx"
        shutil.copy2(exported_onnx_path, dst_onnx_path)
        file_size_mb = dst_onnx_path.stat().st_size / (1024 * 1024)
        print(f"✓ ONNX model successfully saved to: {dst_onnx_path} ({file_size_mb:.2f} MB)")
        
        # Check for Anomalib generated metadata.json
        anomalib_meta = {}
        meta_candidates = list(exported_onnx_path.parent.glob("*.json")) + list(Path(f"./results/{cat}").rglob("metadata.json"))
        if meta_candidates:
            try:
                with open(meta_candidates[0], "r", encoding="utf-8") as mf:
                    anomalib_meta = json.load(mf)
                print(f"✓ Found Anomalib metadata at: {meta_candidates[0]}")
            except Exception as me:
                print(f"Note: Could not parse anomalib metadata.json: {me}")
        
        # Extract threshold and normalization metrics from PyTorch model if available
        img_thresh = None
        pix_thresh = None
        min_score = None
        max_score = None
        
        for candidate_obj in [model, getattr(model, "post_processor", None)]:
            if candidate_obj is None:
                continue
            if hasattr(candidate_obj, "image_threshold"):
                try:
                    val = getattr(candidate_obj.image_threshold, "value", candidate_obj.image_threshold)
                    img_thresh = float(val.cpu().item() if hasattr(val, "cpu") else val)
                except Exception:
                    pass
            if hasattr(candidate_obj, "pixel_threshold"):
                try:
                    val = getattr(candidate_obj.pixel_threshold, "value", candidate_obj.pixel_threshold)
                    pix_thresh = float(val.cpu().item() if hasattr(val, "cpu") else val)
                except Exception:
                    pass
            if hasattr(candidate_obj, "normalization_metrics"):
                try:
                    nm = candidate_obj.normalization_metrics
                    if hasattr(nm, "min"):
                        min_score = float(nm.min.cpu().item() if hasattr(nm.min, "cpu") else nm.min)
                    if hasattr(nm, "max"):
                        max_score = float(nm.max.cpu().item() if hasattr(nm.max, "cpu") else nm.max)
                except Exception:
                    pass
        
        # Fallback to anomalib_meta if direct attributes were not found
        if anomalib_meta:
            if img_thresh is None:
                img_thresh = anomalib_meta.get("image_threshold", anomalib_meta.get("threshold", {}).get("image_threshold"))
            if pix_thresh is None:
                pix_thresh = anomalib_meta.get("pixel_threshold", anomalib_meta.get("threshold", {}).get("pixel_threshold"))
            if min_score is None and "min" in anomalib_meta:
                min_score = anomalib_meta.get("min")
            if max_score is None and "max" in anomalib_meta:
                max_score = anomalib_meta.get("max")

        # Verify model with onnxruntime
        print()
        print(f"[ONNX Runtime Verification for '{cat}']")
        ort_session = ort.InferenceSession(str(dst_onnx_path), providers=["CPUExecutionProvider"])
        onnx_sessions[cat] = ort_session
        
        inputs = ort_session.get_inputs()
        outputs = ort_session.get_outputs()
        
        print(f"Model Input(s) [{len(inputs)}]:")
        for inp in inputs:
            print(f"  - Name: {inp.name}, Shape: {inp.shape}, Type: {inp.type}")
            
        print(f"Model Output(s) [{len(outputs)}]:")
        for out in outputs:
            print(f"  - Name: {out.name}, Shape: {out.shape}, Type: {out.type}")
            
        # Perform a test inference with a dummy input
        primary_input_name = inputs[0].name
        dummy_input = np.zeros((1, 3, 256, 256), dtype=np.float32)
        test_outputs = ort_session.run(None, {primary_input_name: dummy_input})
        
        # Differentiate between anomaly map and anomaly score
        # Anomaly map is a spatial tensor (>=3 dimensions or height/width >= 16)
        # Anomaly score is a scalar or 1D array
        anomaly_map_idx = None
        anomaly_score_idx = None
        
        for idx, out_tensor in enumerate(test_outputs):
            shape = out_tensor.shape
            if len(shape) >= 3 or (len(shape) == 2 and shape[0] >= 16 and shape[1] >= 16):
                anomaly_map_idx = idx
            else:
                anomaly_score_idx = idx
                
        # Fallback to size comparison if heuristics didn't trigger
        if anomaly_map_idx is None or anomaly_score_idx is None or anomaly_map_idx == anomaly_score_idx:
            if test_outputs[0].size > test_outputs[1].size:
                anomaly_map_idx, anomaly_score_idx = 0, 1
            else:
                anomaly_map_idx, anomaly_score_idx = 1, 0
                
        map_out_name = outputs[anomaly_map_idx].name
        map_out_shape = list(test_outputs[anomaly_map_idx].shape)
        score_out_name = outputs[anomaly_score_idx].name
        score_out_shape = list(test_outputs[anomaly_score_idx].shape)
        
        print(f"  ✓ Output '{map_out_name}' identified as: ANOMALY MAP (shape: {map_out_shape})")
        print(f"  ✓ Output '{score_out_name}' identified as: IMAGE SCORE (shape: {score_out_shape})")
        
        # Save comprehensive anomaly_<category>.json
        metadata_content = {
            "category": cat,
            "model_architecture": "EfficientAD-S",
            "onnx_file": f"anomaly_{cat}.onnx",
            "input": {
                "name": primary_input_name,
                "shape": [1, 3, 256, 256],
                "dtype": "float32"
            },
            "preprocessing": {
                "resize": [256, 256],
                "interpolation": "bilinear",
                "scaling": "divide_by_255.0 (maps [0, 255] -> [0.0, 1.0])",
                "normalization": "none (EfficientAd performs teacher/student feature normalization internally)",
                "color_order": "RGB",
                "tensor_format": "NCHW (Batch=1, Channels=3, Height=256, Width=256)"
            },
            "outputs": {
                "anomaly_map": {
                    "name": map_out_name,
                    "shape": map_out_shape,
                    "description": "Pixel-level anomaly heatmap. Higher values indicate higher defect probability."
                },
                "image_score": {
                    "name": score_out_name,
                    "shape": score_out_shape,
                    "description": "Image-level scalar anomaly score."
                }
            },
            "thresholds_and_normalization": {
                "image_threshold": img_thresh,
                "pixel_threshold": pix_thresh,
                "min_score": min_score,
                "max_score": max_score,
                "anomalib_metadata": anomalib_meta
            },
            "metrics": all_metrics.get(cat, {})
        }
        
        dst_json_path = OUTPUT_DIR / f"anomaly_{cat}.json"
        with open(dst_json_path, "w", encoding="utf-8") as jf:
            json.dump(metadata_content, jf, indent=2, ensure_ascii=False)
        print(f"✓ Saved metadata JSON configuration to: {dst_json_path}")
        
    except Exception as e:
        print(f"✗ Failed ONNX export/verification for category '{cat}': {e}")
        import traceback
        traceback.print_exc()


## 5. Conformal Kalibrasyon Skorlarının Hesaplanması
Bu hücrede:
1. Eğitimde **kullanılmamış** (held-out) `test/good` klasöründeki normal/sağlam görseller seçilir (~20 adet; eksik kalırsa `train/good` sonundan tamamlanır).
2. Her görsel ONNX modeli üzerinden geçirilerek görüntü düzeyi anomali skoru hesaplanır.
3. Bu skorlar `out/calib_<kategori>.npy` dosyasına 1-D `float32` dizisi olarak kaydedilir.
4. Bu dosya, sistemin canlı çıkarım esnasında conformal p-değeri üretmesini sağlar:
   $$\text{p-value} = \frac{1 + \sum \mathbf{1}\{s_{\text{cal}} \ge s\}}{N_{\text{cal}} + 1}$$
   - Sağlam numunelerde p-değeri yüksek (~0.5 - 1.0) çıkar.
   - Kusurlu numunelerde skor $s$ yüksek olduğu için p-değeri sıfıra yaklaşır ($p < 0.05$).


In [ ]:
from pathlib import Path
import numpy as np
import onnxruntime as ort
from PIL import Image

for cat in CATEGORIES:
    print()
    print("=" * 70)
    print(f"CONFORMAL CALIBRATION SCORES FOR CATEGORY: {cat}")
    print("=" * 70)
    
    onnx_path = OUTPUT_DIR / f"anomaly_{cat}.onnx"
    json_path = OUTPUT_DIR / f"anomaly_{cat}.json"
    
    if not onnx_path.exists() or not json_path.exists():
        print(f"Skipping calibration for '{cat}' because model or metadata is missing.")
        continue
    
    try:
        # Ensure active onnxruntime session
        session = onnx_sessions.get(cat)
        if session is None:
            session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
            
        with open(json_path, "r", encoding="utf-8") as f:
            meta = json.load(f)
            
        input_name = meta["input"]["name"]
        score_out_name = meta["outputs"]["image_score"]["name"]
        
        # Find held-out good (normal) images
        # Preference 1: test/good (not used during training)
        test_good_dir = DATASET_ROOT / cat / "test" / "good"
        train_good_dir = DATASET_ROOT / cat / "train" / "good"
        
        calib_candidates = []
        if test_good_dir.exists():
            calib_candidates = sorted(list(test_good_dir.glob("*.png")) + list(test_good_dir.glob("*.jpg")))
            print(f"Found {len(calib_candidates)} held-out normal images in '{test_good_dir}'")
            
        # If fewer than 20, supplement with images from train/good (from end)
        if len(calib_candidates) < 20 and train_good_dir.exists():
            train_candidates = sorted(list(train_good_dir.glob("*.png")) + list(train_good_dir.glob("*.jpg")))
            needed = 20 - len(calib_candidates)
            calib_candidates.extend(train_candidates[-needed:])
            print(f"Supplemented with {needed} images from train/good to ensure target size of 20")
            
        if not calib_candidates:
            raise FileNotFoundError(f"No normal/good images found for category '{cat}'.")
            
        # Select ~20 images for calibration
        selected_images = calib_candidates[:20]
        print(f"Running ONNX inference on {len(selected_images)} normal images for calibration...")
        
        scores = []
        for img_p in selected_images:
            with Image.open(img_p) as pil_img:
                # Preprocess: RGB -> resize (256, 256) -> float32 / 255.0 -> NCHW
                img_rgb = pil_img.convert("RGB").resize((256, 256), Image.BILINEAR)
                img_arr = np.array(img_rgb, dtype=np.float32) / 255.0
                img_arr = np.transpose(img_arr, (2, 0, 1))  # HWC -> CHW
                img_tensor = np.expand_dims(img_arr, axis=0)  # [1, 3, 256, 256]
                
            ort_outs = session.run([score_out_name], {input_name: img_tensor})
            raw_score = float(np.ravel(ort_outs[0])[0])
            scores.append(raw_score)
            
        calib_array = np.array(scores, dtype=np.float32)
        calib_dst = OUTPUT_DIR / f"calib_{cat}.npy"
        np.save(calib_dst, calib_array)
        
        print(f"✓ Successfully generated and saved: {calib_dst}")
        print(f"  Shape : {calib_array.shape} (1-D array of float32 scores)")
        print(f"  Dtype : {calib_array.dtype}")
        print(f"  Stats -> Min: {calib_array.min():.5f} | Mean: {calib_array.mean():.5f} | Max: {calib_array.max():.5f} | Std: {calib_array.std():.5f}")
        print(f"  Percentiles -> 25%: {np.percentile(calib_array, 25):.5f} | 50%: {np.percentile(calib_array, 50):.5f} | 75%: {np.percentile(calib_array, 75):.5f} | 90%: {np.percentile(calib_array, 90):.5f}")
        print(f"  [Conformal Note] During production, given anomaly score s, p-value is:")
        print(f"                  p = (1 + sum(s_cal >= s)) / (len(s_cal) + 1)")
        
    except Exception as e:
        print(f"✗ Failed calibration for category '{cat}': {e}")
        import traceback
        traceback.print_exc()


## 6. Çıktıların Arşivlenmesi ve İndirilmesi (ZIP)
Bu hücrede `out/` klasöründeki tüm model dosyaları (`anomaly_<kategori>.onnx`), metaveriler (`anomaly_<kategori>.json`), kalibrasyon dizileri (`calib_<kategori>.npy`) ve test metrikleri (`metrics.json`) listelenir; ardından tek bir ZIP dosyası olarak arşivlenip yerel bilgisayara indirilmesi tetiklenir.


In [ ]:
from pathlib import Path
import shutil

print("=" * 70)
print("ARTIFACTS READY IN out/ DIRECTORY:")
print("=" * 70)

out_files = sorted(list(OUTPUT_DIR.glob("*")))
if not out_files:
    print("No artifacts found in out/!")
else:
    for f in out_files:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  ✓ {f.name:32s} ({size_mb:7.3f} MB)")

# Create zip archive
archive_base = "efficientad_mvtec_out"
zip_path = shutil.make_archive(archive_base, "zip", str(OUTPUT_DIR))
zip_file = Path(zip_path)
print(f"\n✓ Created archive: {zip_file.name} ({zip_file.stat().st_size / (1024 * 1024):.2f} MB)")

# Trigger browser download in Google Colab
try:
    from google.colab import files
    print("\nTriggering file download in Colab browser...")
    files.download(str(zip_file))
    print("✓ Download started.")
except ImportError:
    print("\n[Local Execution Note] google.colab not available.")
    print(f"Archive file is saved at: {zip_file.resolve()}")
